# Ordered Logistic Regression Results for Adoption Predictors: FAIR⁲ Dataset Exploration with `mlcroissant`

This notebook demonstrates step-by-step exploration and processing of the [FAIR⁲ dataset](https://doi.org/10.71728/senscience.y7m0-f273) using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is defined by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

> All dataset structures (record sets, fields, columns) are referenced by their Croissant `@id` for full reproducibility and clarity.

In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant>=0.7.2

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata and schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset Name: ', metadata.name)
print('Dataset Description: ', metadata.description)

## 2. Data Overview

List the available record sets and their `@id`s, then inspect the fields available for each record set.


In [ ]:
# List available record sets in the dataset
record_sets = dataset.record_sets
print("Available record sets (by @id and label):")
for rs in record_sets:
    print(f"  @id: {rs['@id']}, name: {rs.get('name','')}")

# Pick the first record set for further inspection (can select others as needed)
selected_record_set_id = record_sets[0]['@id'] if record_sets else None
print("\nInspecting fields for record set:", selected_record_set_id)

if selected_record_set_id:
    # List the fields for this record set
    record_set = next(rs for rs in record_sets if rs['@id']==selected_record_set_id)
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("Available fields (by @id and name):")
    for f in fields:
        # f can be a @id string or a field dict
        if isinstance(f, dict):
            print(f"  @id: {f['@id']}, name: {f.get('name','')}")
        else:
            print(f"  @id: {f}")
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction

Load data from each record set into a pandas DataFrame for analysis. All entity identifiers (record sets and fields) use their Croissant `@id`. 


In [ ]:
# Load data from each record set using their @id
dfs = {}
for rs in record_sets:
    record_set_id = rs['@id']
    print(f"Loading data for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dfs[record_set_id] = pd.DataFrame(records)
        print(f"  Shape: {dfs[record_set_id].shape}, Columns: {dfs[record_set_id].columns.tolist()}")
    else:
        print(f"  No records found for record set {record_set_id}.")

# For demonstration, pick the first available DataFrame
if dfs:
    main_rs_id = list(dfs.keys())[0]
    print(f"\nColumns for record set {main_rs_id}:")
    print(dfs[main_rs_id].columns.tolist())
    display(dfs[main_rs_id].head())
else:
    print("No record sets could be loaded into dataframes.")

## 4. Exploratory Data Analysis (EDA)

Let's process the loaded records: filter records by a numeric field, normalize values, and optionally group by a category. For all field/column references, `@id` is used as keys in the DataFrame.


In [ ]:
# Pick a record set and inspect its numeric fields (@id)
if dfs:
    df = dfs[main_rs_id]
    print(f"Available columns in record set {main_rs_id}:\n", df.columns.tolist())

    # Attempt to select a numeric field by guessing typical names or picking the first numeric dtype
    import numpy as np
    numeric_candidates = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number) and not pd.isnull(df[col]).all()]
    
    # Fallback if no numeric dtype discovered
    if not numeric_candidates:
        # Try to coerce columns that look numeric
        for col in df.columns:
            try:
                coerced = pd.to_numeric(df[col])
                if not coerced.isnull().all():
                    df[col] = coerced
                    numeric_candidates.append(col)
            except Exception:
                continue

    print(f"Numeric field candidates: {numeric_candidates}")

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Using numeric field @id: {numeric_field_id}")
        threshold = df[numeric_field_id].dropna().median()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
        display(filtered_df.head(3))

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records: (first 5)")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Find a categorical (likely string/object) field to group by
        group_fields = [col for col in df.columns if df[col].dtype==object and col!=numeric_field_id]
        if group_fields:
            group_field = group_fields[0]
            print(f"Grouping by field @id: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame('mean_value')
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field and, if available, compare means across groupings (using their `@id`).

In [ ]:
# Visualization: histogram and group bar plot
import matplotlib.pyplot as plt
import seaborn as sns

if dfs and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(6,4))
    df[numeric_field_id].dropna().hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Barplot for mean by group if available
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(8,4))
        order = df[group_field].value_counts().index[:10]
        # Use filtered_df for better visualization
        sns.barplot(x=group_field, y=numeric_field_id, data=filtered_df, order=order)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.show()
else:
    print("Visualization skipped: numeric field or DataFrame not available.")

## 6. Conclusion

This notebook demonstrated usage of the `mlcroissant` library for programmatic access and analysis of the FAIR⁲ data package on predictors for adoption of indigenous and modern knowledge in rangeland management in Northern Kenya.

- We inspected the Croissant structure and referenced all data entities by their `@id` for maximum reproducibility.
- We loaded record sets and extracted numeric data, demonstrating normalization and grouping with respect to data fields' `@id`.
- Simple visual analytics confirm how a variable's distribution and group means can be interactively explored with minimal code.

For more details, consult the [mlcroissant documentation](https://mlcroissant.readthedocs.io/) or the FAIR⁲ schema at the provided URL.